# VRI 2026 - YOLO Platform Corners Tracker

This notebook unzips `pose_dataset.zip` from your Google Drive, installs Ultralytics, and trains the YOLOv8-Pose model on the ArUco-generated dataset.

In [ ]:
# Mount Google Drive so we can access the dataset and save the .pt weights
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Unzip the dataset and install Ultralytics
!unzip -q -o /content/drive/MyDrive/pose_dataset.zip -d /content/pose_dataset/
!pip install ultralytics
import torch
print(f"Setup complete. Using torch {torch.__version__} ({torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else \"CPU\"})")


In [ ]:
# Train the model natively in Python
from ultralytics import YOLO
import os

yaml_path = "/content/pose_dataset/dataset.yaml"
if not os.path.exists(yaml_path):
    print(f"ERROR: Could not find {yaml_path}. Make sure the dataset was unzipped correctly!")
else:
    # We need to overwrite the path inside dataset.yaml to point to /content/pose_dataset
    with open(yaml_path, "r") as f:
        yaml_content = f.read()
    # Quick hack to fix absolute paths inside dataset.yaml if needed, or rely on relative paths.
    # Since our script wrote absolute paths, we MUST rewrite it for Colab:
    lines = yaml_content.split("\n")
    lines[0] = "path: /content/pose_dataset"
    with open(yaml_path, "w") as f:
        f.write("\n".join(lines))
    
    model = YOLO("yolov8n-pose.pt")
    
    project_dir = "/content/drive/MyDrive/YOLO_Models"
    os.makedirs(project_dir, exist_ok=True)
    
    results = model.train(
        data=yaml_path,
        epochs=100,
        imgsz=640,
        batch=32,
        project=project_dir,
        name="yolov8_platform_corners_v1",
        exist_ok=True,
        
        # --- Augmentations (Extreme to ensure generalization from 1 session) ---
        perspective=0.001,
        fliplr=0.5,
        degrees=90.0,
        translate=0.2,
        scale=0.9,
        mosaic=1.0,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4
    )

    print(f"Training complete! Model saved in {project_dir}/yolov8_platform_corners_v1/weights/best.pt")
